# AegisDesk v2 — GRPO Training Notebook

Train `Qwen3-4B` on all 9 AegisDesk tasks using GRPO (Group Relative Policy Optimization).

**Recommended runtime:** T4 GPU (HF Jobs `t4-medium` @ $0.60/hr, ~2h/round) or Colab Pro T4.

**Steps:**
1. Install dependencies
2. Set credentials
3. Verify environment (dry-run)
4. Run GRPO training
5. Plot reward curves
6. Run self-improvement loop (optional — 3 rounds)


## 1. Install Dependencies

In [ ]:
%%capture
!pip install trl>=0.15.0 transformers>=4.50.0 peft>=0.14.0 bitsandbytes>=0.43.0 \
    accelerate datasets openai httpx pydantic>=2.0 matplotlib

## 2. Set Credentials

In [ ]:
import os

# HF token — get from https://huggingface.co/settings/tokens
os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"

# AegisDesk environment URL (live HF Space)
os.environ["ENV_URL"] = "https://i4mgr00t-meta.hf.space"

# Model to fine-tune
os.environ["MODEL_NAME"] = "Qwen/Qwen3-4B"

# Output directory for checkpoints
os.environ["OUTPUT_DIR"] = "/content/aegisdesk-grpo"

ENV_URL = os.environ["ENV_URL"]
MODEL_NAME = os.environ["MODEL_NAME"]
OUTPUT_DIR = os.environ["OUTPUT_DIR"]

print(f"ENV_URL : {ENV_URL}")
print(f"MODEL   : {MODEL_NAME}")
print(f"OUT DIR : {OUTPUT_DIR}")

## 3. Verify Environment (Dry-Run)

In [ ]:
import httpx, json

ALL_TASKS = [
    "billing_seat_adjustment",
    "login_incident_triage",
    "suspicious_admin_request",
    "customer_escalation_chain",
    "multi_tier_billing_dispute",
    "data_breach_response_lifecycle",
    "contract_renewal_negotiation",
    "service_reinstatement_review",
    "api_partner_access_audit",
]

def check_env(env_url: str):
    """Smoke-test the AegisDesk Space before training."""
    results = {}
    with httpx.Client(timeout=30) as client:
        # Health check
        r = client.get(f"{env_url}/")
        results["health"] = r.status_code == 200

        # Reset on first task
        r = client.post(f"{env_url}/reset", json={"task_id": "billing_seat_adjustment", "seed": 42})
        results["reset"] = r.status_code == 200
        if r.status_code == 200:
            obs = r.json()
            results["inbox_size"] = len(obs.get("inbox", []))

        # Task list — /tasks returns {"tasks": [...]}
        r = client.get(f"{env_url}/tasks")
        if r.status_code == 200:
            data = r.json()
            task_list = data["tasks"] if isinstance(data, dict) else data
            results["task_count"] = len(task_list)
            results["tasks"] = [t["task_id"] for t in task_list]

    return results

results = check_env(ENV_URL)
print(json.dumps(results, indent=2))
assert results.get("health"), "Space health check failed — verify ENV_URL"
assert results.get("reset"), "reset() failed — check Space logs"
print(f"\nEnvironment healthy. {results.get('task_count', '?')} tasks available.")

In [ ]:
# Verify self-improvement pipeline dry-run (no GPU needed)
import subprocess, sys

# Clone repo if running in Colab (skip if already in the repo directory)
if not os.path.exists("training/self_improve.py"):
    !git clone https://github.com/kumarabhik/AegisDesk /content/AegisDesk
    os.chdir("/content/AegisDesk")

result = subprocess.run(
    [sys.executable, "training/self_improve.py",
     "--rounds", "1",
     "--seeds", "42",
     "--env-url", ENV_URL,
     "--dry-run"],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## 4. GRPO Training

Runs `Qwen3-4B` with LoRA 4-bit quantization via TRL `GRPOTrainer`.

**Expected time:** ~2h on T4, ~45min on A10G.

In [ ]:
# Check GPU
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")

In [ ]:
# Full GRPO training run — all 9 tasks, 2 epochs
# Logs reward to OUTPUT_DIR/reward_log.jsonl

!python training/train_grpo_aegisdesk.py \
    --all-tasks \
    --env-url $ENV_URL \
    --model $MODEL_NAME \
    --output-dir $OUTPUT_DIR \
    --num-train-epochs 2 \
    --learning-rate 5e-6 \
    --per-device-train-batch-size 2 \
    --gradient-accumulation-steps 4 \
    --lora-r 16 \
    --lora-alpha 32 \
    --load-in-4bit \
    --seed 42

## 5. Plot Reward Curves

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict
from pathlib import Path

log_path = Path(OUTPUT_DIR) / "reward_log.jsonl"

if not log_path.exists():
    print(f"No reward log found at {log_path}. Run training first.")
else:
    records = [json.loads(l) for l in log_path.read_text().splitlines() if l.strip()]

    # Per-task reward curves
    task_rewards = defaultdict(list)
    task_steps = defaultdict(list)
    overall_rewards = []
    overall_steps = []

    for r in records:
        task = r.get("task_id", "unknown")
        step = r.get("global_step", r.get("step", 0))
        reward = r.get("reward", r.get("mean_reward", 0))
        task_rewards[task].append(reward)
        task_steps[task].append(step)
        overall_rewards.append(reward)
        overall_steps.append(step)

    fig, axes = plt.subplots(3, 3, figsize=(15, 10))
    fig.suptitle("AegisDesk v2 — GRPO Per-Task Reward Curves", fontsize=14, fontweight="bold")

    TASK_ORDER = [
        "billing_seat_adjustment", "login_incident_triage", "suspicious_admin_request",
        "customer_escalation_chain", "multi_tier_billing_dispute", "data_breach_response_lifecycle",
        "contract_renewal_negotiation", "service_reinstatement_review", "api_partner_access_audit",
    ]

    for ax, task in zip(axes.flat, TASK_ORDER):
        steps = task_steps.get(task, [])
        rewards = task_rewards.get(task, [])
        if steps:
            ax.plot(steps, rewards, linewidth=1.5)
            # Rolling mean
            if len(rewards) >= 5:
                window = 5
                rolling = [sum(rewards[max(0,i-window):i+1])/min(window,i+1) for i in range(len(rewards))]
                ax.plot(steps, rolling, linewidth=2, linestyle="--", alpha=0.8, label="rolling mean")
        ax.set_title(task.replace("_", "\n"), fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("training/reward_curves_per_task.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: training/reward_curves_per_task.png")

    # Overall mean reward
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    ax2.plot(overall_steps, overall_rewards, alpha=0.4, linewidth=1, label="per-step")
    if len(overall_rewards) >= 10:
        window = 10
        rolling = [sum(overall_rewards[max(0,i-window):i+1])/min(window,i+1) for i in range(len(overall_rewards))]
        ax2.plot(overall_steps, rolling, linewidth=2, label="rolling mean (10)")
    ax2.axhline(y=0.27, color="red", linestyle=":", label="v1 baseline (0.27)")
    ax2.set_title("AegisDesk v2 — Overall Mean Reward", fontweight="bold")
    ax2.set_xlabel("Training Step")
    ax2.set_ylabel("Reward")
    ax2.set_ylim(0, 1.05)
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("training/reward_curves_overall.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: training/reward_curves_overall.png")

    if overall_rewards:
        final_mean = sum(overall_rewards[-20:]) / min(20, len(overall_rewards))
        print(f"\nFinal 20-step mean reward: {final_mean:.3f}")
        print(f"Improvement over v1 baseline (0.27): {final_mean - 0.27:+.3f}")

## 6. Self-Improvement Loop (3 Rounds)

Runs `harvest → DPO pairs → GRPO → evaluate → delta` three times.

Expected total time: ~6h on T4. Run after initial training checkpoint is saved.

In [ ]:
# Full 3-round self-improvement pipeline
# Remove --dry-run to run actual training

!python training/self_improve.py \
    --rounds 3 \
    --seeds 42,43,44 \
    --env-url $ENV_URL
    # --dry-run  # uncomment to simulate without GPU

## 7. Post-Training Benchmark

Evaluate the trained model against all 9 tasks and compare to the v1 baseline.

In [ ]:
import httpx, json
from openai import OpenAI

HF_TOKEN = os.environ["HF_TOKEN"]

# Use the HF router to run the fine-tuned model
# Replace with your uploaded model ID after pushing to HF Hub
FINETUNED_MODEL = os.environ.get("FINETUNED_MODEL", "Qwen/Qwen3-4B")  # override after push

client = OpenAI(
    api_key=HF_TOKEN,
    base_url="https://router.huggingface.co/v1",
)

SYSTEM_PROMPT = """You are an AI support agent for a B2B SaaS company.
You have access to a support console with tickets, records, and operational actions.
Always inspect relevant records before taking action. Follow policy. Avoid unsafe shortcuts.
Respond with a single JSON object matching the SupportAction schema."""

def run_episode(task_id: str, env_url: str, max_steps: int = 15) -> float:
    """Run one episode and return the final score."""
    with httpx.Client(timeout=60) as http:
        obs = http.post(f"{env_url}/reset", json={"task_id": task_id, "seed": 42}).json()

        for step in range(max_steps):
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Current observation:\n{json.dumps(obs, indent=2)}\n\nWhat action do you take?"}
            ]
            response = client.chat.completions.create(
                model=FINETUNED_MODEL,
                messages=messages,
                temperature=0,
                max_tokens=256,
            )
            action_text = response.choices[0].message.content.strip()

            try:
                action = json.loads(action_text)
            except json.JSONDecodeError:
                import re
                m = re.search(r'\{.*\}', action_text, re.DOTALL)
                action = json.loads(m.group()) if m else {"action_type": "finalize_resolution", "resolution_code": "no_action"}

            step_result = http.post(f"{env_url}/step", json=action).json()
            obs = step_result.get("observation", obs)

            if step_result.get("done"):
                return step_result.get("info", {}).get("final_score", 0.0)

        state = http.post(f"{env_url}/state").json()
        return state.get("rubric_progress", 0.0)


print("Running post-training benchmark...\n")
scores = {}
for task in ALL_TASKS:
    try:
        score = run_episode(task, ENV_URL)
        scores[task] = score
        print(f"  {task:45s}  {score:.3f}")
    except Exception as e:
        scores[task] = 0.0
        print(f"  {task:45s}  ERROR: {e}")

mean = sum(scores.values()) / len(scores)
print(f"\n{'Mean':45s}  {mean:.3f}")
print(f"v1 baseline:                                       0.270")
print(f"Improvement:                                      {mean - 0.27:+.3f}")

## 8. Save Results & Commit

Save the benchmark results and commit plots to the repo.

In [ ]:
import json
from pathlib import Path
from datetime import datetime

results_path = Path("training/benchmark_results.json")
results = {
    "timestamp": datetime.utcnow().isoformat(),
    "model": FINETUNED_MODEL,
    "env_url": ENV_URL,
    "task_scores": scores,
    "mean_score": sum(scores.values()) / len(scores) if scores else 0.0,
    "v1_baseline": 0.27,
}
results_path.write_text(json.dumps(results, indent=2))
print(f"Saved benchmark results to {results_path}")

# Commit plots and results
!git add training/reward_curves_per_task.png training/reward_curves_overall.png training/benchmark_results.json
!git commit -m "Add GRPO training reward curves and benchmark results"
!git push github main
print("Pushed to GitHub.")